![imagen](Airbnb.png)

# Exercise 5

According to our Airbnb project case study, we want to test whether the hypothesis that there are significant price differences between two-bedroom units in the central-west and central-north areas is valid.

## Objectives

1. Construct a confidence interval
2. Perform hypothesis testing
3. Conduct a p-test to compare samples

Airbnb has observed that the central neighborhoods of Centrum-Oost and Centrum-West, despite being in the city center, experience price fluctuations that tend to affect the number of bookings from one area to another. Airbnb asks you to determine whether the price differences observed between the central-east (Centrum-Oost) and central-west (Centrum-West) areas are significant. They are particularly focused on comparing two-bedroom apartments.


# 1. Import libraries

In [1]:
import os
import pandas as pd
import numpy as np
# To calculate the statistical test
from statsmodels.stats import weightstats as stests


# 2. Load data

In [31]:
data= pd.read_csv('listings_m-2-.csv', sep=',')

In [32]:
#To display all columns
pd.set_option('display.max_columns', None)
data.head(2)

,id,listing_url,scrape_id,last_scraped,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2818,https://www.airbnb.com/rooms/2818,2.021040e+13,12/04/2021,Quiet Garden View Room & Super Fast WiFi,Quiet Garden View Room & Super Fast WiFi<br />...,"Indische Buurt (""Indies Neighborhood"") is a ne...",https://a0.muscache.com/pictures/10272854/8dcc...,3159,https://www.airbnb.com/users/show/3159,Daniel,24/09/2008,"Amsterdam, Noord-Holland, The Netherlands","Upon arriving in Amsterdam, one can imagine as...",NaN,NaN,NaN,t,https://a0.muscache.com/im/users/3159/profile_...,https://a0.muscache.com/im/users/3159/profile_...,Indische Buurt,1.0,1.0,"['email', 'phone', 'reviews', 'jumio', 'offlin...",t,t,"Amsterdam, North Holland, Netherlands",Oostelijk Havengebied - Indische Buurt,NaN,"5,236,435","494,358",Private room in apartment,Private room,2,NaN,1.5 shared baths,1.0,2.0,"[""Long term stays allowed"", ""Wifi"", ""Hot water...",59,3,1125,3.0,3.0,1125.0,1125.0,3.0,1125.0,NaN,t,13,43,62,152,12/04/2021,278,0,0,30/03/2009,14/02/2020,98.0,10.0,10.0,10.0,10.0,9.0,10.0,NaN,t,1,0,1,0,1.9
1,20168,https://www.airbnb.com/rooms/20168,2.021040e+13,12/04/2021,Studio with private bathroom in the centre 1,17th century Dutch townhouse in the heart of t...,Located just in between famous central canals....,https://a0.muscache.com/pictures/69979628/fd6a...,59484,https://www.airbnb.com/users/show/59484,Alexander,02/12/2009,"Amsterdam, Noord-Holland, The Netherlands",#¿NOMBRE?,NaN,NaN,NaN,f,https://a0.muscache.com/im/pictures/user/65092...,https://a0.muscache.com/im/pictures/user/65092...,Grachtengordel,2.0,2.0,"['email', 'phone', 'reviews', 'jumio', 'offlin...",t,t,"Amsterdam, North Holland, Netherlands",Centrum-Oost,NaN,"5,236,407","489,393",Private room in townhouse,Private room,2,NaN,1 private bath,1.0,1.0,"[""Essentials"", ""TV"", ""Host greets you"", ""Wifi""...",200,1,365,1.0,4.0,365.0,1125.0,3.3,554.1,NaN,t,0,0,0,0,12/04/2021,339,0,0,02/03/2010,09/04/2020,89.0,10.0,10.0,10.0,10.0,10.0,9.0,0363 CBB3 2C10 0C2A 1E29,t,2,0,2,0,2.5


# 3. Confidence interval
To calculate it, you need: 
* 𝑋 = Sample mean
* 𝛼 = Significance
* 𝜎 = Standard deviation
* 𝑛 = Sample size
* 𝑧 = Z-test value

In [39]:
x=data['price'].mean()
alpha=0.05 
sigma=data['price'].std()
n=len(data['price'])
z=1.96
standard_error = sigma /np.sqrt(n)
lower_limit = x - z* standard_error  
upper_limit = x + z* standard_error
interval=[lower_limit, upper_limit]
print(f"Confidence interval: [{lower_limit:.2f}, {upper_limit:.2f}]")

Confidence interval: [152.23, 156.76]


#### The confidence interval for prices of all properties available on the platform in Amsterdam ranges from 152.23 to 156.76 euros.

# 4. Hypothesis testing
* H0: The prices are the same
* H1: Prices are different 

In [45]:
# Filter the variable data I'll need for the test and sort them based on the required criteria (properties located in the ‘Centrum-Oost’ and ‘Centrum-West’ neighborhoods with 2 bedrooms).
neighbourhood = data[
    (data['room_type'] == 'Entire home/apt') & #Compare apartments only
    (data['bedrooms'] == 2) & #2-bedroom
    (data['neighbourhood_cleansed'].isin(['Centrum-Oost', 'Centrum-West'])) #In the Centrum-Oost and Centrum-West neighborhoods
][['neighbourhood_cleansed', 'room_type', 'price', 'bedrooms']]
neighbourhood.head()

,neighbourhood_cleansed,room_type,price,bedrooms
7,Centrum-West,Entire home/apt,211,2.0
10,Centrum-West,Entire home/apt,157,2.0
73,Centrum-West,Entire home/apt,125,2.0
74,Centrum-West,Entire home/apt,541,2.0
78,Centrum-Oost,Entire home/apt,385,2.0


In [46]:
# Delete null values
neighbourhood=neighbourhood.dropna()

In [47]:
Oost=neighbourhood[neighbourhood['neighbourhood_cleansed']=='Centrum-Oost']['price']
West=neighbourhood[neighbourhood['neighbourhood_cleansed']=='Centrum-West']['price']
print('Mean Centrum-Oost: '+str(Oost.mean()))
print('Mean Centrum-West: '+str(West.mean()))

Mean Centrum-Oost: 237.61514195583595
Mean Centrum-West: 275.6100278551532


In [52]:
#Z-test
ztest ,pval1 = stests.ztest(West, x2=Oost, value=0,alternative='two-sided')
print('Z value: '+str(ztest))
print('P value: '+ str(pval1))

Z value: 1.46724270456459
P value: 0.1423100437108674


In [55]:
confidence=0.95
significance=1-confidence
if pval1<significance:
    print("Reject the null hypothesis")
else:
    print("Do not reject the null hypothesis")

Do not reject the null hypothesis


#### At a 95% confidence level, there is not enough statistical evidence to conclude that there is a significant difference between the prices of two-bedroom apartments in Centrum-Oost and Centrum-West.